# Task 3: TwinLiteNetPlus Baseline on NightCity

Ultra-lightweight real-time segmentation model trained from scratch and evaluated on NightCity (19 classes, `ignore_index=255`).

### Architectural Summary
- **Model:** `TwinLiteNetPlus` (0.45M params, Depthwise ESP blocks + CAAM attention module + multi-stage decoder, ~415+ FPS).
- **Objective:** OHEM Cross-Entropy semantic loss:
  $$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{sem}}$$
- **NightCity Benchmark:** Serves as the ultra-lightweight baseline against which Knowledge Distillation (Option 2) is audited.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# Auto-detect if running on Kaggle
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()

if IS_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yacs", "thop"])
    repo_dest = Path("/kaggle/working/AI512_Project")
    if not repo_dest.exists():
        subprocess.run(["git", "clone", "https://github.com/Gitansh-Bansal/AI512_Project.git", str(repo_dest)])
    else:
        subprocess.run(["git", "-C", str(repo_dest), "pull"])
    os.chdir(str(repo_dest))

# Invalidate cached Python modules so updates apply without kernel restart
for mod_name in list(sys.modules.keys()):
    if any(prefix in mod_name for prefix in ['train_nightcity', 'datasets.nightcity', 'datasets.base_dataset', 'model.twinlitenet_seg', 'twinlitenet_seg']):
        del sys.modules[mod_name]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

CWD = Path(".").resolve()
REPO_ROOT = next((p for p in [Path(".").resolve()] + list(Path(".").resolve().parents) if (p / "TwinLiteNetPlus").exists()), Path(".").resolve())
TWINLITE_DIR = REPO_ROOT / "TwinLiteNetPlus"

# Auto-locate NightCity dataset (searches Kaggle input first, then local)
DATA_ROOT = None
if Path("/kaggle/input").exists():
    items = list(Path("/kaggle/input").iterdir())
    print(f"[INFO] Detected in /kaggle/input: {[p.name for p in items]}")
    for f in Path("/kaggle/input").rglob("Chicago_*.png"):
        for p_cand in f.parents:
            has_images = (p_cand / "NightCity-images").exists() or (p_cand / "images").exists()
            has_labels = (p_cand / "NightCity-label").exists() or (p_cand / "label").exists()
            if has_images and has_labels:
                DATA_ROOT = p_cand
                break
        if DATA_ROOT:
            break

    if DATA_ROOT is None:
        for f in Path("/kaggle/input").rglob("Chicago_*.png"):
            p_cand = f.parent
            while p_cand != Path("/kaggle/input") and p_cand != Path("/"):
                if p_cand.name.lower() in ("nightcity", "nightcity-dataset"):
                    DATA_ROOT = p_cand
                    break
                p_cand = p_cand.parent
            if DATA_ROOT:
                break

    if DATA_ROOT is None:
        for cand in [
            Path("/kaggle/input/datasets/gitanshb/nightcity"),
            Path("/kaggle/input/nightcity"),
            Path("/kaggle/input/nightcity-dataset"),
        ]:
            if cand.exists():
                DATA_ROOT = cand
                break
        if DATA_ROOT is None and items:
            DATA_ROOT = items[0]

if DATA_ROOT is not None:
    while DATA_ROOT.name.lower() in ("nightcity-label", "nightcity-images", "label", "images"):
        DATA_ROOT = DATA_ROOT.parent

if DATA_ROOT is None:
    DATA_ROOT = (REPO_ROOT / "Data") if (REPO_ROOT / "Data").exists() else (REPO_ROOT / "AI512_Project" / "Data")

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"TWINLITE_DIR: {TWINLITE_DIR}")
print(f"DATA_ROOT: {DATA_ROOT}")

if str(TWINLITE_DIR) not in sys.path:
    sys.path.insert(0, str(TWINLITE_DIR))

from model.twinlitenet_seg import get_twinlitenet_plus_seg
from datasets.nightcity import NightCity
from metrics import SegmentationMetrics, CITYSCAPES_CLASSES
from benchmarks import get_hardware_info, profile_parameters, profile_flops, benchmark_inference_speed
from train_nightcity import run_training, plot_training_history

CITYSCAPES_PALETTE = np.array([
    [128, 64, 128], [244, 35, 232], [70, 70, 70], [102, 102, 156], [190, 153, 153],
    [153, 153, 153], [250, 170, 30], [220, 220, 0], [107, 142, 35], [152, 251, 152],
    [70, 130, 180], [220, 20, 60], [255, 0, 0], [0, 0, 142], [0, 0, 70],
    [0, 60, 100], [0, 80, 100], [0, 0, 230], [119, 11, 32]
], dtype=np.uint8)

def colorize_mask(mask: np.ndarray) -> np.ndarray:
    """Colorizes a 2D integer segmentation mask into Cityscapes RGB palette."""
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for c in range(min(19, len(CITYSCAPES_PALETTE))):
        color_mask[mask == c] = CITYSCAPES_PALETTE[c]
    return color_mask

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch: {torch.__version__}")


## 1. Model, Complexity & Speed

Inspect TwinLiteNetPlus parameters, FLOPs, and hardware inference speed.

In [ ]:
model = get_twinlitenet_plus_seg(config='small', num_classes=19).to(device).eval()

with torch.no_grad():
    out = model(torch.randn(1, 3, 512, 1024, device=device))
print("Input: (1, 3, 512, 1024) -> Output:", out.shape)

params = profile_parameters(model)
flops = profile_flops(model, [(512, 1024)], device=device)
speed = benchmark_inference_speed(model, input_resolution=(512, 1024), batch_size=1, device=device, warmup_runs=10, timed_runs=30)

hw = get_hardware_info(device)
hw_name = hw.get('GPU_Device_Name', hw.get('CPU_Processor', 'Unknown'))
latency = speed.get('Mean_Latency_ms', speed.get('mean_latency_ms', 0.0))
print(f"Hardware: {hw_name}")
print(f"Params:   {params['total_params_M']:.2f} M")
print(f"FLOPs (512x1024):  {flops[0]['FLOPs_G']:.2f} GFLOPs")
print(f"Speed:    {speed['FPS']:.2f} FPS | Latency: {latency:.2f} ms")


## 2. Baseline Training (NightCity)

Train TwinLiteNetPlus on NightCity for 30 epochs using AMP FP16, AdamW optimizer, and OHEM Cross-Entropy loss.

In [ ]:
train_list_p = TWINLITE_DIR / "data" / "list" / "nightcity" / "train.lst"
val_list_p = TWINLITE_DIR / "data" / "list" / "nightcity" / "val.lst"

TRAIN_CONFIG = {
    "epochs": 30,
    "batch_size": 4 if torch.cuda.is_available() else 2,
    "optimizer": "adamw",
    "base_lr": 0.001,
    "model_config": "small",
    "crop_size": (512, 1024),
    "val_freq": 2,
    "val_samples": None if torch.cuda.is_available() else 10,
    "max_train_samples": None if torch.cuda.is_available() else 8,
    "save_dir": str(REPO_ROOT / "output" / "nightcity_twinlitenetplus_train"),
}

history = run_training(
    data_root=str(DATA_ROOT),
    train_list=str(train_list_p),
    val_list=str(val_list_p),
    model_config=TRAIN_CONFIG["model_config"],
    epochs=TRAIN_CONFIG["epochs"],
    batch_size=TRAIN_CONFIG["batch_size"],
    optimizer_name=TRAIN_CONFIG["optimizer"],
    base_lr=TRAIN_CONFIG["base_lr"],
    crop_h=TRAIN_CONFIG["crop_size"][0],
    crop_w=TRAIN_CONFIG["crop_size"][1],
    val_freq=TRAIN_CONFIG["val_freq"],
    val_samples=TRAIN_CONFIG["val_samples"],
    num_train_samples=TRAIN_CONFIG["max_train_samples"],
    save_dir=TRAIN_CONFIG["save_dir"],
)

fig = plot_training_history(history, save_path=str(REPO_ROOT / "output" / "nightcity_twinlitenetplus_train" / "loss_curves.png"))
plt.show()


## 3. Evaluation (NightCity Validation)

Evaluate the best TwinLiteNetPlus baseline checkpoint across the validation set and collect 5 distinct scenes for visual inspection.

In [ ]:
EVAL_SAMPLES = None if torch.cuda.is_available() else 100

val_ds = NightCity(
    root=str(DATA_ROOT),
    list_path=str(val_list_p),
    num_classes=19, multi_scale=False, flip=False, crop_size=(512, 1024),
)
if EVAL_SAMPLES:
    val_ds.files = val_ds.files[:EVAL_SAMPLES]
val_loader = DataLoader(val_ds, batch_size=4 if torch.cuda.is_available() else 2, shuffle=False)

ckpt_path = REPO_ROOT / "output" / "nightcity_twinlitenetplus_train" / "best_twinlitenetplus_nightcity.pt"
if not ckpt_path.exists():
    ckpt_path = REPO_ROOT / "output" / "nightcity_twinlitenetplus_train" / "latest_twinlitenetplus_nightcity.pt"

eval_model = get_twinlitenet_plus_seg(config='small', num_classes=19).to(device)
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt.get("state_dict", ckpt)
    eval_model.load_state_dict({k.replace("module.", ""): v for k, v in state.items()}, strict=False)
    print(f"[INFO] Loaded checkpoint from: {ckpt_path}")
eval_model.eval()

metrics = SegmentationMetrics(num_classes=19, ignore_label=255, class_names=CITYSCAPES_CLASSES)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
vis_samples = []

with torch.no_grad():
    for imgs, lbls, _, _, names in val_loader:
        preds = eval_model(imgs.to(device))
        if preds.shape[-2:] != lbls.shape[-2:]:
            preds = F.interpolate(preds, size=lbls.shape[-2:], mode="bilinear", align_corners=True)
        pred_cls = torch.argmax(preds, dim=1).cpu().numpy()
        metrics.update(pred_cls, lbls.numpy())

        for b in range(imgs.shape[0]):
            if len(vis_samples) < 5:
                rgb = np.clip((imgs[b].permute(1, 2, 0).numpy() * std + mean) * 255, 0, 255).astype(np.uint8)
                vis_samples.append((names[b], rgb, lbls[b].numpy(), pred_cls[b]))

summary = metrics.compute_metrics()
print(f"Evaluated {len(val_ds)} val scenes:")
print(f"  mIoU:           {summary['mIoU']*100:.2f}%")
print(f"  Mean Dice (F1): {summary['mean_Dice']*100:.2f}%")
print(f"  Sensitivity:    {summary['mean_Sensitivity']*100:.2f}%")
print(f"  Specificity:    {summary['mean_Specificity']*100:.2f}%")
print(f"  Pixel Accuracy: {summary['Pixel_Accuracy']*100:.2f}%")


## 4. Per-Class Breakdown & Visual Predictions

Inspect per-class IoU and visualize 5 distinct validation scenes (Input RGB, Ground Truth, TwinLiteNetPlus Prediction).

In [ ]:
df_cls = pd.DataFrame(summary["Per_Class"])
for col in ["IoU", "Dice", "Sensitivity", "Specificity"]:
    df_cls[f"{col} (%)"] = (df_cls.pop(col) * 100).round(2)
display(df_cls[["Class_ID", "Class_Name", "Pixels_GT", "IoU (%)", "Dice (%)", "Sensitivity (%)", "Specificity (%)"]])

fig, axes = plt.subplots(len(vis_samples), 3, figsize=(15, 3.8 * len(vis_samples)))
for i, (name, rgb, gt, pred) in enumerate(vis_samples):
    axes[i, 0].imshow(rgb); axes[i, 0].set_title(name); axes[i, 0].axis("off")
    axes[i, 1].imshow(colorize_mask(gt)); axes[i, 1].set_title("Ground Truth"); axes[i, 1].axis("off")
    axes[i, 2].imshow(colorize_mask(pred)); axes[i, 2].set_title("TwinLiteNetPlus Prediction"); axes[i, 2].axis("off")
plt.tight_layout()
plt.show()


## 5. Model Overview & NightCity Baseline Summary

Summary of TwinLiteNetPlus architectural parameters and NightCity nocturnal baseline performance.

In [ ]:
hw_type = "GPU" if device.type == "cuda" else "CPU"
display(pd.DataFrame({
    "Metric": ["Dataset", "Architecture", "Params", "FLOPs (512x1024)", "Inference Speed", "Validation mIoU", "Pixel Accuracy"],
    "TwinLiteNetPlus NightCity Baseline": [
        "NightCity",
        "Depthwise ESPNet + CAAM Attention",
        "0.45 M",
        "~5.8 G",
        "~415+ FPS",
        f"{summary['mIoU']*100:.2f}% mIoU",
        f"{summary['Pixel_Accuracy']*100:.2f}%"
    ]
}))
